In [1]:
!rm -rf /kaggle/working/*

In [2]:
import warnings
import numpy as np
import pandas as pd
import lightgbm as lgb
import xgboost as xgb
from catboost import CatBoostRegressor
from sklearn.model_selection import KFold
from sklearn.metrics import root_mean_squared_log_error

warnings.filterwarnings('ignore')

In [3]:
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)
pd.set_option('display.width', 1000)
pd.set_option('display.precision', 4)
pd.set_option('display.max_colwidth', None)

In [4]:
train=pd.read_csv("/kaggle/input/competitions/green-grid-forecasting-daily-household-energy-consumption/train.csv")

test=pd.read_csv("/kaggle/input/competitions/green-grid-forecasting-daily-household-energy-consumption/test.csv")

sample_sub=pd.read_csv("/kaggle/input/competitions/green-grid-forecasting-daily-household-energy-consumption/sample_submission.csv")

In [5]:
train.head()

,row_id,household_id,date,num_residents,home_sqft,has_ev,has_solar,has_pool,heating_type,hvac_age_years,temp_avg_c,temp_min_c,temp_max_c,humidity_pct,wind_kph,precip_mm,solar_index,is_weekend,is_holiday,prior_day_kwh,prior_week_avg_kwh,kwh
0,0,H00000,2025-01-02,4,1654,0,0,0,electric,25,-6.5,-11.0,-2.0,70,12.9,0.0,7.4,0,0,29.11,29.1100,32.91
1,1,H00000,2025-01-03,4,1654,0,0,0,electric,25,-6.5,-9.0,-3.9,37,10.1,0.0,6.3,0,0,32.91,31.0100,27.97
2,2,H00000,2025-01-04,4,1654,0,0,0,electric,25,-6.7,-9.0,-4.5,92,9.9,0.0,7.2,1,0,27.97,29.9967,30.81
3,3,H00000,2025-01-05,4,1654,0,0,0,electric,25,-7.0,-11.0,-3.1,65,15.1,0.0,3.3,1,0,30.81,30.2000,32.56
4,4,H00000,2025-01-06,4,1654,0,0,0,electric,25,-7.3,-10.9,-3.8,71,17.0,10.2,7.2,0,0,32.56,30.6720,31.93


In [6]:
test.head()

,row_id,household_id,date,num_residents,home_sqft,has_ev,has_solar,has_pool,heating_type,hvac_age_years,temp_avg_c,temp_min_c,temp_max_c,humidity_pct,wind_kph,precip_mm,solar_index,is_weekend,is_holiday,prior_day_kwh,prior_week_avg_kwh
0,59,H00000,2025-03-02,4,1654,0,0,0,electric,25,2.2,-0.3,4.6,32,8.8,0.0,10.0,1,0,27.82,25.0029
1,60,H00000,2025-03-03,4,1654,0,0,0,electric,25,2.3,-1.6,6.2,59,4.8,12.6,4.8,0,0,20.67,24.4886
2,61,H00000,2025-03-04,4,1654,0,0,0,electric,25,2.1,-2.6,6.9,87,18.3,1.4,6.5,0,0,20.12,24.3329
3,62,H00000,2025-03-05,4,1654,0,0,0,electric,25,2.3,-0.2,4.7,32,7.9,0.0,9.0,0,0,27.89,24.9457
4,63,H00000,2025-03-06,4,1654,0,0,0,electric,25,2.3,-2.1,6.6,60,13.9,0.0,7.6,0,0,21.79,25.0786


In [7]:
sample_sub.head()

,row_id,kwh
0,59,25.26
1,60,25.26
2,61,25.26
3,62,25.26
4,63,25.26


In [8]:
sample_sub.shape

(21000, 2)

In [9]:
train.shape

(88500, 22)

In [10]:
train.isnull().sum()

row_id                0
household_id          0
date                  0
num_residents         0
home_sqft             0
has_ev                0
has_solar             0
has_pool              0
heating_type          0
hvac_age_years        0
temp_avg_c            0
temp_min_c            0
temp_max_c            0
humidity_pct          0
wind_kph              0
precip_mm             0
solar_index           0
is_weekend            0
is_holiday            0
prior_day_kwh         0
prior_week_avg_kwh    0
kwh                   0
dtype: int64

In [11]:
test.shape

(21000, 21)

In [12]:
test.isnull().sum()

row_id                0
household_id          0
date                  0
num_residents         0
home_sqft             0
has_ev                0
has_solar             0
has_pool              0
heating_type          0
hvac_age_years        0
temp_avg_c            0
temp_min_c            0
temp_max_c            0
humidity_pct          0
wind_kph              0
precip_mm             0
solar_index           0
is_weekend            0
is_holiday            0
prior_day_kwh         0
prior_week_avg_kwh    0
dtype: int64

In [13]:
import pandas as pd
import numpy as np
import lightgbm as lgb
import xgboost as xgb
from catboost import CatBoostRegressor
from sklearn.model_selection import KFold
from sklearn.linear_model import Ridge
from sklearn.metrics import root_mean_squared_log_error
import warnings
warnings.filterwarnings('ignore')

# Assuming train and test are already loaded
train['date'] = pd.to_datetime(train['date'])
test['date'] = pd.to_datetime(test['date'])

def engineer_features(df):
    df['day_of_week'] = df['date'].dt.dayofweek
    df['day_of_month'] = df['date'].dt.day
    df['month'] = df['date'].dt.month
    df['is_weekend_int'] = df['is_weekend'].astype(int)
    
    df['cdd'] = (df['temp_avg_c'] - 18).clip(lower=0)
    df['hdd'] = (18 - df['temp_avg_c']).clip(lower=0)
    df['temp_range'] = df['temp_max_c'] - df['temp_min_c']
    
    df['hvac_cooling_load'] = df['cdd'] * df['home_sqft']
    df['hvac_heating_load'] = df['hdd'] * df['home_sqft']

    df['cooling_inefficiency'] = df['hvac_cooling_load'] * df['hvac_age_years']
    df['heating_inefficiency'] = df['hvac_heating_load'] * df['hvac_age_years']
    
    df['solar_power_potential'] = df['has_solar'] * df['solar_index'] * df['home_sqft']
    
    df['resident_density'] = df['home_sqft'] / df['num_residents'].clip(lower=1)
    df['base_load_proxy'] = df['num_residents'] * df['home_sqft']
    
    return df

train = engineer_features(train)
test = engineer_features(test)

# Household statistics
household_stats = train.groupby('household_id')['kwh'].agg(['median', 'std']).reset_index()
household_stats.columns = ['household_id', 'hh_median_kwh', 'hh_std_kwh']

train = train.merge(household_stats, on='household_id', how='left')
test = test.merge(household_stats, on='household_id', how='left')

train['hh_std_kwh'] = train['hh_std_kwh'].fillna(0)
test['hh_std_kwh'] = test['hh_std_kwh'].fillna(0)

# Categorical columns
cat_cols = ['heating_type', 'household_id']
for col in cat_cols:
    train[col] = train[col].astype('category')
    test[col] = test[col].astype('category')

print("Feature engineering completed!")

Feature engineering completed!


In [14]:
features = [c for c in train.columns if c not in ['row_id', 'date', 'kwh']]
target = 'kwh'

X = train[features]
y = train[target]
X_test = test[features]

y_log = np.log1p(y)

kf = KFold(n_splits=5, shuffle=True, random_state=42)

oof_preds = pd.DataFrame(index=X.index, columns=['lgb', 'xgb', 'cat'])
test_preds = pd.DataFrame(index=X_test.index, columns=['lgb', 'xgb', 'cat']).fillna(0.0)

cat_features = list(X.select_dtypes(include=['category']).columns)

lgb_params = {'objective': 'regression', 'metric': 'rmse', 'learning_rate': 0.03, 
              'max_depth': 7, 'num_leaves': 63, 'colsample_bytree': 0.8, 
              'subsample': 0.8, 'random_state': 42, 'verbose': -1, 'n_estimators': 2000}

xgb_params = {'objective': 'reg:squarederror', 'eval_metric': 'rmse', 'learning_rate': 0.03, 
              'max_depth': 6, 'colsample_bytree': 0.8, 'subsample': 0.8, 
              'random_state': 42, 'n_estimators': 2000, 'tree_method': 'hist', 
              'enable_categorical': True}

cat_params = {'loss_function': 'RMSE', 'learning_rate': 0.04, 'depth': 6, 
              'random_seed': 42, 'verbose': 0, 'iterations': 2000, 
              'early_stopping_rounds': 100}

for fold, (train_idx, val_idx) in enumerate(kf.split(X, y_log)):
    print(f"--- Training Fold {fold + 1} ---")
    X_train, y_train = X.iloc[train_idx], y_log.iloc[train_idx]
    X_val, y_val = X.iloc[val_idx], y_log.iloc[val_idx]
    
    # LightGBM
    model_lgb = lgb.LGBMRegressor(**lgb_params)
    model_lgb.fit(X_train, y_train, eval_set=[(X_val, y_val)], 
                  callbacks=[lgb.early_stopping(100, verbose=False)])
    oof_preds.loc[val_idx, 'lgb'] = model_lgb.predict(X_val)
    test_preds['lgb'] += model_lgb.predict(X_test) / kf.n_splits
    
    # XGBoost
    model_xgb = xgb.XGBRegressor(**xgb_params)
    model_xgb.fit(X_train, y_train, eval_set=[(X_val, y_val)], verbose=False)
    oof_preds.loc[val_idx, 'xgb'] = model_xgb.predict(X_val)
    test_preds['xgb'] += model_xgb.predict(X_test) / kf.n_splits
    
    # CatBoost
    model_cat = CatBoostRegressor(**cat_params)
    model_cat.fit(X_train, y_train, eval_set=(X_val, y_val), 
                  cat_features=cat_features, verbose=False)
    oof_preds.loc[val_idx, 'cat'] = model_cat.predict(X_val)
    test_preds['cat'] += model_cat.predict(X_test) / kf.n_splits

print("Base models training completed!")

--- Training Fold 1 ---
--- Training Fold 2 ---
--- Training Fold 3 ---
--- Training Fold 4 ---
--- Training Fold 5 ---
Base models training completed!


In [15]:
print("\n--- Training Meta-Model (Ridge Regression) ---")
meta_model = Ridge(alpha=1.0)
meta_model.fit(oof_preds, y_log)

print("Learned Model Weights (LGB, XGB, CAT):", meta_model.coef_)

final_oof_log = meta_model.predict(oof_preds)
final_test_log = meta_model.predict(test_preds)

final_oof_preds = np.clip(np.expm1(final_oof_log), 0, None)
final_test_preds = np.clip(np.expm1(final_test_log), 0, None)

cv_score = root_mean_squared_log_error(y, final_oof_preds)
print(f"\nFinal Stacked Out-Of-Fold RMSLE: {cv_score:.4f}")

# Create submission
sample_sub = pd.DataFrame({'row_id': test['row_id'], 'kwh': final_test_preds})
sample_sub.to_csv('submission.csv', index=False)
print("Submission saved successfully as 'submission.csv'!")


--- Training Meta-Model (Ridge Regression) ---
Learned Model Weights (LGB, XGB, CAT): [-0.08973267  0.03564667  1.05471794]

Final Stacked Out-Of-Fold RMSLE: 0.1509
Submission saved successfully as 'submission.csv'!
